# 00 · Data, k-space, and the forward operator

Goal: load a magnitude slice, move between image and k-space with the centred
FFT, build a Cartesian undersampling mask, and simulate an undersampled
measurement `y = M ⊙ F(x)`.

**Prereq:** follow `data/REGISTER_FIRST.md`, then `pixi run download && pixi run preprocess`.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from mrigen.data import FastMRISlices
from mrigen.fourier import fft2c, ifft2c
from mrigen import viz

In [ ]:
ds = FastMRISlices('../data/processed')
x = jnp.asarray(ds[0])
print('image', x.shape, 'range', float(x.min()), float(x.max()))
viz.show_image(x, title='magnitude slice'); plt.show()

## k-space round trip
`fourier.py` is given. One FFT convention everywhere — this is the #1 silent-bug source.

In [ ]:
k = fft2c(x)
viz.show_kspace(k); plt.show()
back = ifft2c(k).real
print('round-trip error', float(jnp.abs(back - x).max()))

## Build a mask and undersample  *(TODO: `masks.py`, Team B)*

In [ ]:
from mrigen.masks import equispaced_mask
mask = equispaced_mask((128, 128), acceleration=4)
y = mask * fft2c(x)              # undersampled measurement
from mrigen.recon.classical import zero_filled
x_zf = zero_filled(y, mask)
viz.panel(x, x_zf); plt.show()